# 2-1. 단일표본 t-검정과 독립표본 t-검정

- 단일표본 t-검정으로 성인 여성 키의 평균이 기준값 163cm와 다른지 확인한다.
- 독립표본 t-검정으로 A반과 B반의 평균 점수 차이를 확인한다.

## 이 실습의 학습 기준

이 노트북은 한 가지 함수의 결과만 확인하지 않고 다음 흐름으로 학습한다.

1. 데이터 구조에 맞는 검정 방법과 가정을 확인한다.
2. 같은 목적의 방법이 여러 개면 모두 실행해 결과를 비교한다.
3. 일부러 적합하지 않은 방법도 비교할 때는, 왜 최종 결론에 사용하지 않는지 밝힌다.
4. 전제검정 결과가 본 검정 선택에 어떻게 연결되는지 확인한다.
5. p-value뿐 아니라 차이의 방향과 크기까지 해석한다.
6. p-value가 `0.0000`으로 보이는 것은 반올림 결과이며, 실제 확률이 0이라는 뜻은 아니다.
7. 유의수준 α=0.05에서 `p-value < 0.05`면 귀무가설을 기각하고, `p-value >= 0.05`면 귀무가설을 기각하지 못한다.

In [1]:
import numpy as np          # 평균, 표준편차 등 수치 계산
import pandas as pd         # CSV 파일을 표 형태로 읽기
from scipy import stats     # t-검정, 정규성 검정 등 통계 함수

In [2]:
# 텍스트 파일을 읽기 모드("r")로 연다.
with open("datas2/성인여성_키_데이터.txt", "r") as file:
    # 빈 줄이 있어도 안전하게 처리하기 위해 splitlines()를 사용한다.
    height_text = file.read().splitlines()

# 문자열로 읽힌 키 데이터를 실수(float) 리스트로 변환한다.
heights = list(map(float, height_text))

print("앞에서 5개:", heights[:5])
print("데이터 개수:", len(heights))

앞에서 5개: [150.27, 142.94, 160.99, 157.48, 153.46]
데이터 개수: 25


## 1. 단일표본 t-검정

성인 여성 키 표본의 평균이 기준값 163cm와 통계적으로 다른지 확인한다.

- 귀무가설(H₀): 성인 여성 키의 평균은 163cm이다.
- 대립가설(H₁): 성인 여성 키의 평균은 163cm와 다르다.

In [3]:
# 표본 키 데이터의 평균과 표본 표준편차를 계산한다.
height_mean = np.mean(heights)
height_std = np.std(heights, ddof=1)

print(f"평균 키: {height_mean:.2f}cm")
print(f"표본 표준편차: {height_std:.2f}cm")

평균 키: 156.93cm
표본 표준편차: 10.18cm


### 1-1. KS-test와 Shapiro-Wilk 정규성 검정 비교

단일표본 t-검정 전에 키 데이터가 정규분포에서 크게 벗어나지 않는지 확인한다.

- KS-test: 실제 데이터의 누적분포와 기준 정규분포의 누적분포를 비교한다.
- Shapiro-Wilk: 정렬된 데이터의 형태가 정규분포에서 기대되는 형태와 비슷한지 확인한다.
- 귀무가설(H₀): 키 데이터는 정규분포를 따른다.
- 대립가설(H₁): 키 데이터는 정규분포를 따르지 않는다.

여기서는 두 방법을 모두 경험하고 결과를 비교한다. 다만 아래 KS-test는 표본에서 계산한 평균과 표준편차를 기준분포에 사용하므로 학습용 비교다. 작은 표본의 정규성 판단은 Shapiro-Wilk 결과를 중심으로 살펴본다.

In [4]:
# Shapiro-Wilk 정규성 검정을 수행한다.
height_shapiro = stats.shapiro(heights)

print(f"Shapiro-Wilk 검정통계량: {height_shapiro.statistic:.4f}")
print(f"p-value: {height_shapiro.pvalue:.4f}")

Shapiro-Wilk 검정통계량: 0.9536
p-value: 0.3014


In [5]:
# KS-test로 표본의 누적분포와 정규분포의 누적분포를 비교한다.
height_ks = stats.kstest(
    heights,
    stats.norm.cdf,
    args=(height_mean, height_std)
)

print(f"KS 검정통계량: {height_ks.statistic:.4f}")
print(f"p-value: {height_ks.pvalue:.4f}")

KS 검정통계량: 0.1122
p-value: 0.8772


In [6]:
# 두 정규성 검정의 결과를 같은 표에서 비교한다.
height_normality_comparison = pd.DataFrame({
    "검정 방법": ["KS-test", "Shapiro-Wilk"],
    "검정통계량": [height_ks.statistic, height_shapiro.statistic],
    "p-value": [height_ks.pvalue, height_shapiro.pvalue]
})

height_normality_comparison["결론"] = height_normality_comparison["p-value"].apply(
    lambda p: "정규성 위반 증거 부족" if p >= 0.05 else "정규성 위반 가능성"
)

display(height_normality_comparison.round(4))

same_height_conclusion = (height_ks.pvalue >= 0.05) == (height_shapiro.pvalue >= 0.05)
print("두 검정의 결론 일치 여부:", same_height_conclusion)
print("본 실습의 주 판단: Shapiro-Wilk 결과를 중심으로 해석한다.")

,검정 방법,검정통계량,p-value,결론
0,KS-test,0.1122,0.8772,정규성 위반 증거 부족
1,Shapiro-Wilk,0.9536,0.3014,정규성 위반 증거 부족


두 검정의 결론 일치 여부: True
본 실습의 주 판단: Shapiro-Wilk 결과를 중심으로 해석한다.


### 1-2. 단일표본 t-검정 수행

정규성 검정 결과를 확인한 뒤, 유의수준 0.05에서 성인 여성 키의 평균이 163cm와 다른지 양측 검정한다.

In [7]:
reference_mean = 163

# 표본평균을 기준값 163cm와 비교한다.
one_sample_result = stats.ttest_1samp(heights, popmean=reference_mean)

print(f"t-통계량: {one_sample_result.statistic:.4f}")
print(f"p-value: {one_sample_result.pvalue:.4f}")

if one_sample_result.pvalue < 0.05:
    print("귀무가설을 기각한다: 평균 키는 163cm와 통계적으로 다르다.")
else:
    print("귀무가설을 기각하지 못한다: 평균 키가 163cm와 다르다고 할 충분한 증거가 없다.")

t-통계량: -2.9798
p-value: 0.0065
귀무가설을 기각한다: 평균 키는 163cm와 통계적으로 다르다.


## 2. 독립표본 t-검정

A반과 B반은 서로 다른 학생으로 구성된 독립된 두 그룹이다.

- 귀무가설(H₀): A반과 B반의 평균 점수는 같다.
- 대립가설(H₁): A반과 B반의 평균 점수는 다르다.

In [8]:
scores_df = pd.read_csv("datas2/반별_점수_type1.csv", encoding="euc-kr")
scores_df.head()

,반,점수
0,A,73
1,A,69
2,A,71
3,A,71
4,A,73


In [9]:
group_a = scores_df.loc[scores_df["반"] == "A", "점수"].to_numpy()
group_b = scores_df.loc[scores_df["반"] == "B", "점수"].to_numpy()

print("A반 인원:", len(group_a), "점수 일부:", group_a[:5])
print("B반 인원:", len(group_b), "점수 일부:", group_b[:5])

A반 인원: 20 점수 일부: [73 69 71 71 73]
B반 인원: 10 점수 일부: [63 56 73 61 55]


In [10]:
mean_a = np.mean(group_a)
mean_b = np.mean(group_b)
std_a = np.std(group_a, ddof=1)
std_b = np.std(group_b, ddof=1)

print(f"A반 평균: {mean_a:.2f}, 표준편차: {std_a:.2f}")
print(f"B반 평균: {mean_b:.2f}, 표준편차: {std_b:.2f}")
print(f"평균 차이(A반 - B반): {mean_a - mean_b:.2f}")

A반 평균: 70.55, 표준편차: 5.68
B반 평균: 64.10, 표준편차: 8.28
평균 차이(A반 - B반): 6.45


### 2-1. A반과 B반의 정규성 검정 비교

각 반에 KS-test와 Shapiro-Wilk를 모두 적용한다. KS-test는 방법을 경험하기 위한 비교이고, 표본 수가 작은 A반과 B반의 주 판단은 Shapiro-Wilk를 사용한다.

- 귀무가설(H₀): 해당 반의 점수는 정규분포를 따른다.
- 대립가설(H₁): 해당 반의 점수는 정규분포를 따르지 않는다.

In [11]:
group_normality_rows = []

for group_name, group_values in [("A반", group_a), ("B반", group_b)]:
    group_mean = np.mean(group_values)
    group_std = np.std(group_values, ddof=1)
    ks_result = stats.kstest(
        group_values,
        stats.norm.cdf,
        args=(group_mean, group_std)
    )
    shapiro_result = stats.shapiro(group_values)

    group_normality_rows.extend([
        {
            "그룹": group_name,
            "검정 방법": "KS-test",
            "검정통계량": ks_result.statistic,
            "p-value": ks_result.pvalue
        },
        {
            "그룹": group_name,
            "검정 방법": "Shapiro-Wilk",
            "검정통계량": shapiro_result.statistic,
            "p-value": shapiro_result.pvalue
        }
    ])

group_normality_comparison = pd.DataFrame(group_normality_rows)
group_normality_comparison["결론"] = group_normality_comparison["p-value"].apply(
    lambda p: "정규성 위반 증거 부족" if p >= 0.05 else "정규성 위반 가능성"
)

display(group_normality_comparison.round(4))

,그룹,검정 방법,검정통계량,p-value,결론
0,A반,KS-test,0.1331,0.8255,정규성 위반 증거 부족
1,A반,Shapiro-Wilk,0.9697,0.7485,정규성 위반 증거 부족
2,B반,KS-test,0.1588,0.9295,정규성 위반 증거 부족
3,B반,Shapiro-Wilk,0.8888,0.1646,정규성 위반 증거 부족


### 2-2. Levene 등분산성 검정

독립표본 t-검정에는 두 가지 방식이 있다.

- Student 독립표본 t-검정: 두 그룹의 분산이 같다고 가정한다.
- Welch 독립표본 t-검정: 두 그룹의 분산이 같다고 가정하지 않는다.

Levene 검정으로 어떤 방식을 주 결론에 사용할지 정한다.

- 귀무가설(H₀): A반과 B반의 점수 분산은 같다.
- 대립가설(H₁): A반과 B반의 점수 분산은 다르다.

In [12]:
levene_result = stats.levene(group_a, group_b)

print(f"Levene 검정통계량: {levene_result.statistic:.4f}")
print(f"p-value: {levene_result.pvalue:.4f}")

equal_variance = levene_result.pvalue >= 0.05
print("Student t-검정을 주 결과로 사용:", equal_variance)

Levene 검정통계량: 2.0331
p-value: 0.1650
Student t-검정을 주 결과로 사용: True


### 2-3. Student t-검정과 Welch t-검정 비교

두 방식을 모두 실행해 분산 가정에 따라 t-통계량, 자유도, p-value가 어떻게 달라지는지 확인한다. 최종 결론에는 앞의 Levene 결과로 선택한 방식을 사용한다.

In [13]:
student_result = stats.ttest_ind(group_a, group_b, equal_var=True)
welch_result = stats.ttest_ind(group_a, group_b, equal_var=False)

independent_comparison = pd.DataFrame({
    "검정 방법": ["Student t-test", "Welch t-test"],
    "등분산 가정": [True, False],
    "t-통계량": [student_result.statistic, welch_result.statistic],
    "자유도": [student_result.df, welch_result.df],
    "p-value": [student_result.pvalue, welch_result.pvalue]
})
independent_comparison["결론"] = independent_comparison["p-value"].apply(
    lambda p: "평균 차이 유의" if p < 0.05 else "평균 차이 증거 부족"
)

display(independent_comparison.round(4))

if equal_variance:
    selected_test_name = "Student 독립표본 t-검정"
    independent_result = student_result
else:
    selected_test_name = "Welch 독립표본 t-검정"
    independent_result = welch_result

print("Levene 검정에 따라 선택한 방법:", selected_test_name)
print(f"선택한 검정의 p-value: {independent_result.pvalue:.4f}")

,검정 방법,등분산 가정,t-통계량,자유도,p-value,결론
0,Student t-test,True,2.5129,28.0000,0.0180,평균 차이 유의
1,Welch t-test,False,2.2166,13.3832,0.0445,평균 차이 유의


Levene 검정에 따라 선택한 방법: Student 독립표본 t-검정
선택한 검정의 p-value: 0.0180


### 2-4. 독립표본 t-검정 결과

- A반 평균은 70.55점, B반 평균은 64.10점으로 차이는 6.45점이었다.
- KS-test와 Shapiro-Wilk 모두 두 반에서 정규성 위반의 충분한 증거를 보이지 않았다.
- Levene 검정 p-value는 0.1650이므로 Student 독립표본 t-검정을 주 결과로 선택했다.
- Student 검정의 p-value는 0.0180, Welch 검정의 p-value는 0.0445였다.
- 두 방식 모두 유의수준 0.05에서 평균 차이가 유의하다는 같은 결론을 보였지만 통계량과 p-value는 달랐다.
- 주 결과에 따라 A반과 B반의 평균 점수는 통계적으로 다르며, A반 평균이 더 높다고 해석한다.

## 3. 같은 데이터의 저장 형태 비교

`type1`은 반 이름과 점수가 행으로 쌓인 긴 형식이고, `type2`는 A반과 B반이 별도 열인 넓은 형식이다. 저장 형태가 달라도 같은 표본인지 확인한다.

In [14]:
scores_wide_df = pd.read_csv(
    "datas2/반별_점수_type2.csv",
    encoding="euc-kr"
)
scores_wide_df.head()

,A반,B반
0,73,63.0
1,69,56.0
2,71,73.0
3,71,61.0
4,73,55.0


In [15]:
group_a_wide = scores_wide_df["A반"].dropna().to_numpy()
group_b_wide = scores_wide_df["B반"].dropna().to_numpy()

print("A반 데이터 동일 여부:", np.array_equal(group_a, group_a_wide))
print("B반 데이터 동일 여부:", np.array_equal(group_b, group_b_wide))

A반 데이터 동일 여부: True
B반 데이터 동일 여부: True
